# Segment profile: products per user, credit limit, current balance and credit score

**Goal.** Descriptive statistics per customer `segment` (Basic, Plus, Premium, Student) built from
`data/data/customers.csv` and `data/data/products.csv`: **average products per user, credit limit, current
balance and credit score**.

**Definitions used (all stated here so the table can be read without guessing):**

- **Segment** is `customers.segment`; a product belongs to the segment of its owner (`customer_id`).
- **Products per user** counts every product (all types and statuses). It is reported over *all* customers in the
  segment (customers with no product count as 0) and over customers with at least one product.
- **Credit limit** exists only on **credit products** (cards, personal loans, mortgages; 95% of them have a value), so its
  statistics use those products only.
- **Current balance** is shown for all products, and separately for credit products (a debt) and deposit accounts (savings and
  checking, an asset), because mixing them in one average would not mean anything.
- **Currencies.** Balances and limits are in ARS, COP or USD and the file has no exchange rates. They are converted to
  **USD-equivalent using the rates implied by the transactions file** (`amount / amount_usd`), an assumption read off the
  data, not a market rate.
- **Credit score** comes from `customers.credit_score`; customers without a score (missing) are left out of its statistics and their share is reported.

Descriptive only, no model. No row-level customer data is displayed.

## 1. Setup and load

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIRECTORY = REPO_ROOT / "data" / "data"
TRANSACTIONS_JUNE_2026 = REPO_ROOT / "data" / "transactions" / "year=2026" / "month=06"

SEGMENT_ORDER = ["Basic", "Plus", "Premium", "Student"]
CREDIT_PRODUCT_TYPES = ["Tarjeta Crédito", "Préstamo Personal", "Préstamo Hipotecario"]
DEPOSIT_PRODUCT_TYPES = ["Cuenta Ahorro", "Cuenta Corriente"]

customers = pd.read_csv(DATA_DIRECTORY / "customers.csv", usecols=["customer_id", "segment", "credit_score"])
products = pd.read_csv(
    DATA_DIRECTORY / "products.csv",
    usecols=["product_id", "customer_id", "product_type", "product_status", "currency", "current_balance", "credit_limit"],
)
display(pd.Series({
    "customers": len(customers),
    "products": len(products),
    "products whose owner is not in customers.csv": (~products["customer_id"].isin(customers["customer_id"])).sum(),
    "customers with no product": (~customers["customer_id"].isin(products["customer_id"])).sum(),
    "customers without a segment": customers["segment"].isna().sum(),
}, name="value").to_frame())

,value
customers,150000
products,400000
products whose owner is not in customers.csv,0
customers with no product,10422
customers without a segment,0


### Exchange rates implied by the transactions file

In [2]:
june_files = sorted(TRANSACTIONS_JUNE_2026.glob("day=*/transactions_*.csv"))
recorded = pd.concat(
    (pd.read_csv(path, usecols=["currency", "amount", "amount_usd"]) for path in june_files), ignore_index=True
).dropna(subset=["amount_usd"])
recorded["units_per_usd"] = recorded["amount"] / recorded["amount_usd"]

UNITS_PER_USD = recorded.groupby("currency")["units_per_usd"].median().round(2).to_dict()
UNITS_PER_USD["USD"] = 1.0
display(recorded.groupby("currency")["units_per_usd"].describe())
UNITS_PER_USD

,count,mean,std,min,25%,50%,75%,max
currency,,,,,,,,
ARS,"12,034.00",350.00,0.01,349.70,350.00,350.00,350.00,350.19
COP,"18,169.00","4,000.00",0.13,"3,997.13","3,999.99","4,000.00","4,000.01","4,003.27"


{'ARS': 350.0, 'COP': 4000.0, 'USD': 1.0}

## 2. Prepare the analysis frames

In [3]:
products["is_credit"] = products["product_type"].isin(CREDIT_PRODUCT_TYPES)
products["is_deposit"] = products["product_type"].isin(DEPOSIT_PRODUCT_TYPES)
units_per_usd = products["currency"].map(UNITS_PER_USD)
products["balance_usd"] = products["current_balance"] / units_per_usd
products["credit_limit_usd"] = products["credit_limit"] / units_per_usd

per_customer = products.groupby("customer_id").agg(
    products=("product_id", "size"),
    credit_products=("is_credit", "sum"),
    total_credit_limit_usd=("credit_limit_usd", "sum"),
)
credit_balance_per_customer = (
    products[products["is_credit"]].groupby("customer_id")["balance_usd"].sum().rename("total_credit_balance_usd")
)
customer_view = (
    customers.merge(per_customer, on="customer_id", how="left")
    .merge(credit_balance_per_customer, on="customer_id", how="left")
)
customer_view[["products", "credit_products"]] = customer_view[["products", "credit_products"]].fillna(0)
customer_view["segment"] = pd.Categorical(customer_view["segment"], SEGMENT_ORDER, ordered=True)

product_view = products.merge(customers[["customer_id", "segment"]], on="customer_id", how="left")
product_view["segment"] = pd.Categorical(product_view["segment"], SEGMENT_ORDER, ordered=True)

display(pd.Series({
    "credit products": product_view["is_credit"].sum(),
    "  with a credit_limit": product_view.loc[product_view["is_credit"], "credit_limit"].notna().sum(),
    "  without a credit_limit (left out of limit statistics)": product_view.loc[product_view["is_credit"], "credit_limit"].isna().sum(),
    "non-credit products with a credit_limit": product_view.loc[~product_view["is_credit"], "credit_limit"].notna().sum(),
}, name="value").to_frame())

,value
credit products,131972
with a credit_limit,125317
without a credit_limit (left out of limit statistics),6655
non-credit products with a credit_limit,0


## 3. Customer level: products per user and credit score

In [4]:
def describe_by_segment(values: pd.Series, segment: pd.Series, statistics: list[str]) -> pd.DataFrame:
    """Computes descriptive statistics of one measure for every segment and for all customers.

    Exists so every measure in the segment table is summarised in the same way, including a total row.

    Args:
        values: The measure, aligned to `segment`; missing values are ignored.
        segment: Segment of each row.
        statistics: Names of the aggregations to compute (for example `mean`, `median`).

    Returns:
        pd.DataFrame: one row per segment plus an `All` row, one column per statistic.
    """
    by_segment = values.groupby(segment, observed=True).agg(statistics)
    everyone = values.agg(statistics).to_frame("All").T
    return pd.concat([by_segment, everyone])


STATISTICS = ["count", "mean", "median", "std", "min", "max"]

products_per_user_all = describe_by_segment(customer_view["products"], customer_view["segment"], STATISTICS)
has_product = customer_view["products"] > 0
products_per_user_with_product = describe_by_segment(
    customer_view.loc[has_product, "products"], customer_view.loc[has_product, "segment"], STATISTICS
)
display(products_per_user_all.rename_axis("products per user, ALL customers (0 if none)"))
display(products_per_user_with_product.rename_axis("products per user, customers with >= 1 product"))
display(describe_by_segment(customer_view["credit_products"], customer_view["segment"], ["mean", "median", "std", "max"])
        .rename_axis("credit products per user, ALL customers"))

,count,mean,median,std,min,max
"products per user, ALL customers (0 if none)",,,,,,
Basic,"89,756.00",2.67,2.00,1.63,0.00,11.00
Plus,"37,547.00",2.68,3.00,1.64,0.00,12.00
Premium,"15,207.00",2.64,2.00,1.61,0.00,10.00
Student,"7,490.00",2.67,2.00,1.66,0.00,11.00
All,"150,000.00",2.67,2.00,1.63,0.00,12.00


,count,mean,median,std,min,max
"products per user, customers with >= 1 product",,,,,,
Basic,"83,514.00",2.87,3.00,1.52,1.00,11.00
Plus,"34,966.00",2.88,3.00,1.52,1.00,12.00
Premium,"14,125.00",2.84,3.00,1.48,1.00,10.00
Student,"6,973.00",2.87,3.00,1.54,1.00,11.00
All,"139,578.00",2.87,3.00,1.52,1.00,12.00


,mean,median,std,max
"credit products per user, ALL customers",,,,
Basic,0.88,1.00,0.94,8.00
Plus,0.88,1.00,0.94,6.00
Premium,0.87,1.00,0.93,6.00
Student,0.87,1.00,0.94,6.00
All,0.88,1.00,0.94,8.00


In [5]:
credit_score_stats = describe_by_segment(customer_view["credit_score"], customer_view["segment"], STATISTICS)
credit_score_stats["missing_pct"] = pd.concat([
    customer_view["credit_score"].isna().groupby(customer_view["segment"], observed=True).mean() * 100,
    pd.Series({"All": customer_view["credit_score"].isna().mean() * 100}),
])
display(credit_score_stats.rename_axis("credit score (customers with a score)"))

,count,mean,median,std,min,max,missing_pct
credit score (customers with a score),,,,,,,
Basic,"76,281.00",599.54,600.00,39.96,422.00,780.00,15.01
Plus,"31,848.00",699.18,699.00,39.79,542.00,850.00,15.18
Premium,"13,020.00",797.40,799.00,36.68,640.00,850.00,14.38
Student,"6,359.00",649.02,649.00,39.96,503.00,779.00,15.10
All,"127,508.00",647.10,631.00,76.85,422.00,850.00,14.99


## 4. Product level: credit limit and current balance (USD-equivalent)

In [6]:
credit_limit_stats = describe_by_segment(
    product_view.loc[product_view["is_credit"], "credit_limit_usd"],
    product_view.loc[product_view["is_credit"], "segment"], STATISTICS)
balance_all = describe_by_segment(product_view["balance_usd"], product_view["segment"], STATISTICS)
balance_credit = describe_by_segment(
    product_view.loc[product_view["is_credit"], "balance_usd"], product_view.loc[product_view["is_credit"], "segment"], STATISTICS)
balance_deposit = describe_by_segment(
    product_view.loc[product_view["is_deposit"], "balance_usd"], product_view.loc[product_view["is_deposit"], "segment"], STATISTICS)

display(credit_limit_stats.rename_axis("credit limit per credit product (USD-eq)"))
display(balance_all.rename_axis("current balance, ALL products (USD-eq)"))
display(balance_credit.rename_axis("current balance, CREDIT products (USD-eq)"))
display(balance_deposit.rename_axis("current balance, DEPOSIT accounts (USD-eq)"))

,count,mean,median,std,min,max
credit limit per credit product (USD-eq),,,,,,
Basic,"75,020.00","38,038.17","30,581.14","32,649.77","1,000.33","149,996.05"
Plus,"31,520.00","37,934.33","30,507.23","32,469.84","1,000.57","149,981.30"
Premium,"12,596.00","37,619.71","30,430.31","32,226.09","1,003.15","149,953.35"
Student,"6,181.00","38,875.72","31,294.33","33,299.14","1,003.12","149,971.64"
All,"125,317.00","38,011.30","30,588.19","32,595.07","1,000.33","149,996.05"


,count,mean,median,std,min,max
"current balance, ALL products (USD-eq)",,,,,,
Basic,"239,337.00","5,610.41","2,436.79","15,061.58",0.00,"216,201.46"
Plus,"100,544.00","5,559.22","2,434.23","14,854.53",0.00,"216,517.28"
Premium,"40,084.00","5,654.76","2,416.43","15,338.14",0.00,"220,377.89"
Student,"20,035.00","5,537.59","2,473.22","14,726.07",0.00,"185,631.01"
All,"400,000.00","5,598.34","2,435.79","15,021.17",0.00,"220,377.89"


,count,mean,median,std,min,max
"current balance, CREDIT products (USD-eq)",,,,,,
Basic,"78,975.00","9,619.56","1,779.39","25,366.69",0.00,"216,201.46"
Plus,"33,184.00","9,458.48","1,787.97","25,007.99",0.00,"216,517.28"
Premium,"13,287.00","9,734.50","1,797.96","25,776.41",0.00,"220,377.89"
Student,"6,526.00","9,534.17","1,776.88","24,950.33",0.00,"185,631.01"
All,"131,972.00","9,586.41","1,782.91","25,298.06",0.00,"220,377.89"


,count,mean,median,std,min,max
"current balance, DEPOSIT accounts (USD-eq)",,,,,,
Basic,"131,868.00","3,879.64","3,462.30","2,347.28",0.00,"14,896.24"
Plus,"55,307.00","3,873.62","3,446.63","2,356.58",0.00,"15,843.39"
Premium,"21,887.00","3,880.02","3,441.43","2,348.72",0.00,"13,880.99"
Student,"11,120.00","3,885.47","3,464.73","2,361.81",0.00,"14,045.63"
All,"220,182.00","3,878.46","3,456.97","2,350.49",0.00,"15,843.39"


In [7]:
customers_with_credit = customer_view[customer_view["credit_products"] > 0]
per_customer_totals = pd.concat({
    "total credit limit per customer with credit (USD-eq)":
        describe_by_segment(customers_with_credit["total_credit_limit_usd"], customers_with_credit["segment"], ["count", "mean", "median", "std"]),
    "total credit balance per customer with credit (USD-eq)":
        describe_by_segment(customers_with_credit["total_credit_balance_usd"], customers_with_credit["segment"], ["count", "mean", "median", "std"]),
}, axis=1)
display(per_customer_totals)

total credit limit per customer with credit (USD-eq)                               total credit balance per customer with credit (USD-eq)                             
                                                       count      mean    median       std                                                  count      mean   median       std
Basic                                            52,490.00   54,365.09 40,113.46 48,634.28                                          52,490.00     14,473.32 2,536.24 32,052.36
Plus                                             22,042.00   54,245.99 40,044.22 48,632.84                                          22,042.00     14,239.64 2,539.00 31,467.21
Premium                                           8,903.00   53,224.52 39,552.93 48,085.64                                           8,903.00     14,527.94 2,531.87 32,553.96
Student                                           4,335.00   55,430.41 40,342.85 51,084.96                                           4,335.00     14,352.93 2,570.20 31,768.16
All                                              87,770.00   54,272.10 40,056.46 48,703.65                                          87,770.00     14,414.23 2,538.41 31,943.50

### The same metrics by product type

The averages above blend cards, loans and mortgages, whose sizes differ by orders of magnitude, so a segment's
average depends on its product mix as much as on its customers. By product type:

In [8]:
credit_by_type = (
    product_view[product_view["is_credit"]]
    .groupby(["product_type", "segment"], observed=True)
    .agg(products=("product_id", "size"), mean_limit_usd=("credit_limit_usd", "mean"),
         median_limit_usd=("credit_limit_usd", "median"), mean_balance_usd=("balance_usd", "mean"),
         median_balance_usd=("balance_usd", "median"))
)
display(credit_by_type)
share_by_type = pd.crosstab(product_view.loc[product_view["is_credit"], "segment"],
                            product_view.loc[product_view["is_credit"], "product_type"], normalize="index") * 100
display(share_by_type.rename_axis("share of credit products by type (%)"))

products  mean_limit_usd  median_limit_usd  mean_balance_usd  median_balance_usd
product_type         segment                                                                                  
Préstamo Hipotecario Basic        7163       77,280.19         78,258.02         80,140.26           79,814.73
                     Plus         2983       76,017.93         75,021.72         78,880.16           79,120.20
                     Premium      1192       75,491.53         73,288.16         82,107.96           81,712.41
                     Student       572       80,538.90         83,845.74         81,268.21           80,587.60
Préstamo Personal    Basic       11895       77,634.97         78,297.12          8,004.78            7,915.08
                     Plus         5063       77,020.33         77,712.80          8,002.42            7,916.53
                     Premium      1997       76,017.96         75,319.88          8,087.14            8,143.57
                     Student      1005       79,012.35         79,601.24          8,181.82            8,286.28
Tarjeta Crédito      Basic       59917       25,475.41         25,533.88          1,509.48            1,499.78
                     Plus        25138       25,554.90         25,536.30          1,513.81            1,505.09
                     Premium     10098       25,508.59         25,450.28          1,517.09            1,511.56
                     Student      4949       25,949.79         25,929.31          1,517.85            1,495.04

product_type,Préstamo Hipotecario,Préstamo Personal,Tarjeta Crédito
share of credit products by type (%),,,
Basic,9.07,15.06,75.87
Plus,8.99,15.26,75.75
Premium,8.97,15.03,76.00
Student,8.76,15.40,75.84


## 5. Summary table

In [9]:
segments_and_all = SEGMENT_ORDER + ["All"]

summary = pd.DataFrame({
    "customers": pd.concat([customer_view.groupby("segment", observed=True).size(), pd.Series({"All": len(customer_view)})]),
    "products per user (all customers): mean": products_per_user_all["mean"],
    "products per user (all customers): median": products_per_user_all["median"],
    "products per user (with >= 1 product): mean": products_per_user_with_product["mean"],
    "credit limit per credit product, USD-eq: mean": credit_limit_stats["mean"],
    "credit limit per credit product, USD-eq: median": credit_limit_stats["median"],
    "current balance per product (all), USD-eq: mean": balance_all["mean"],
    "current balance per product (all), USD-eq: median": balance_all["median"],
    "credit score: mean": credit_score_stats["mean"],
    "credit score: median": credit_score_stats["median"],
    "credit score: std": credit_score_stats["std"],
    "credit score missing %": credit_score_stats["missing_pct"],
}).loc[segments_and_all]
display(summary.T)
summary.to_csv(Path.cwd() / "segment_profile_summary.csv", index_label="segment")

,Basic,Plus,Premium,Student,All
customers,"89,756.00","37,547.00","15,207.00","7,490.00","150,000.00"
products per user (all customers): mean,2.67,2.68,2.64,2.67,2.67
products per user (all customers): median,2.00,3.00,2.00,2.00,2.00
products per user (with >= 1 product): mean,2.87,2.88,2.84,2.87,2.87
"credit limit per credit product, USD-eq: mean","38,038.17","37,934.33","37,619.71","38,875.72","38,011.30"
"credit limit per credit product, USD-eq: median","30,581.14","30,507.23","30,430.31","31,294.33","30,588.19"
"current balance per product (all), USD-eq: mean","5,610.41","5,559.22","5,654.76","5,537.59","5,598.34"
"current balance per product (all), USD-eq: median","2,436.79","2,434.23","2,416.43","2,473.22","2,435.79"
credit score: mean,599.54,699.18,797.40,649.02,647.10
credit score: median,600.00,699.00,799.00,649.00,631.00


## 6. How different are the segments? (95% intervals of the means)

In [10]:
def mean_with_interval(values: pd.Series, segment: pd.Series) -> pd.DataFrame:
    """Computes each segment's mean and the 95% half-width of that mean.

    Exists so differences between segments can be judged against their noise.

    Args:
        values: The measure; missing values are ignored.
        segment: Segment of each row.

    Returns:
        pd.DataFrame: one row per segment with count, mean and 95% half-width.
    """
    grouped = values.groupby(segment, observed=True).agg(count="count", mean="mean", std="std")
    grouped["ci_95"] = 1.96 * grouped["std"] / grouped["count"] ** 0.5
    return grouped.drop(columns="std")


credit_rows = product_view["is_credit"]
intervals = pd.concat({
    "products per user (all customers)": mean_with_interval(customer_view["products"], customer_view["segment"]),
    "credit limit per credit product (USD-eq)": mean_with_interval(product_view.loc[credit_rows, "credit_limit_usd"], product_view.loc[credit_rows, "segment"]),
    "current balance per product (USD-eq)": mean_with_interval(product_view["balance_usd"], product_view["segment"]),
    "credit score": mean_with_interval(customer_view["credit_score"], customer_view["segment"]),
}, axis=0)
display(intervals)

count      mean  ci_95
                                         segment                         
products per user (all customers)        Basic     89756      2.67   0.01
                                         Plus      37547      2.68   0.02
                                         Premium   15207      2.64   0.03
                                         Student    7490      2.67   0.04
credit limit per credit product (USD-eq) Basic     75020 38,038.17 233.64
                                         Plus      31520 37,934.33 358.46
                                         Premium   12596 37,619.71 562.79
                                         Student    6181 38,875.72 830.16
current balance per product (USD-eq)     Basic    239337  5,610.41  60.34
                                         Plus     100544  5,559.22  91.82
                                         Premium   40084  5,654.76 150.16
                                         Student   20035  5,537.59 203.91
credit score                             Basic     76281    599.54   0.28
                                         Plus      31848    699.18   0.44
                                         Premium   13020    797.40   0.63
                                         Student    6359    649.02   0.98

## 7. Reading the table

- **Only the credit score separates the segments.** Mean scores are 599.5 (Basic), 649.0 (Student), 699.2 (Plus) and
  797.4 (Premium), with a standard deviation of about 40 inside each segment (36.7 for Premium) and intervals of
  +/-0.3 to +/-1.0 points. About 15% of every segment has no score.
- **Everything else is the same in every segment**, within the 95% intervals of section 6:
  - Products per user: 2.64-2.68 over all customers (2.84-2.88 over customers with at least one product). The median of
    Plus is 3 against 2 elsewhere only because the means (2.67-2.68) sit right at the edge between two whole numbers.
  - Credit limit per credit product: 37,620-38,876 USD-equivalent (mean), 30,430-31,294 (median); intervals +/-234 to +/-830.
  - Current balance per product: 5,538-5,655 USD-equivalent (mean), 2,416-2,473 (median).
  - The product mix is also the same (about 76% cards, 15% personal loans, 9% mortgages in every segment), and the
    per-type limits and balances match across segments.
- So `Premium` and `Plus` customers do **not** hold more products, larger limits or larger balances than `Basic`. In
  this data the segment behaves as a credit-score band, not as a wealth or engagement tier.

**Caveats.**

- Amounts are USD-equivalent at the rates implied by the transactions file (350 ARS/USD, 4,000 COP/USD), an assumption
  read off the data.
- 6,655 of the 131,972 credit products have no `credit_limit` and are left out of the limit statistics.
- Means are much higher than medians for limits and balances (for example a balance mean of 5.6 k against a
  median of 2.4 k), because mortgages are far larger than cards; medians describe the typical product better.
- Segments were taken as recorded in `customers.csv`; the summary table is exported to `segment_profile_summary.csv`.